# Task 1: Rating Prediction via Prompting

## Objectives
1. Load `yelp_ratings.csv`.
2. **Sample ~250 rows** for evaluation.
3. Implement **6 Prompting Strategies**: 
    - Zero-shot Baseline
    - Few-shot
    - Chain-of-Thought
    - Strict JSON Enforcer
    - Self-Correction/Retry
    - **Self-Consistency (Majority Vote)**
4. Evaluate Accuracy, JSON Validity, and Reliability.

## Setup
To use real LLM calls, set the `OPENAI_API_KEY` environment variable. Defaults to mock if not found.

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "sk-..." # Replace with your actual key

In [1]:
import pandas as pd
import os
import json
from sklearn.metrics import accuracy_score

# Load Data
if os.path.exists('yelp_ratings.csv'):
    df = pd.read_csv('yelp_ratings.csv')
    print("Loaded dataset: yelp_ratings.csv")
else:
    raise FileNotFoundError("yelp_ratings.csv not found. Please upload the dataset.")

# Normalize columns
df.columns = df.columns.str.lower().str.strip()
rename_map = {
    'rating': 'stars', 'class index': 'stars',
    'review': 'text', 'review text': 'text', 'desc': 'text', 'description': 'text'
}
df.rename(columns=rename_map, inplace=True)

# --- SAMPLING STEP ---
TARGET_SAMPLE_SIZE = 250
if len(df) > TARGET_SAMPLE_SIZE:
    df = df.sample(n=TARGET_SAMPLE_SIZE, random_state=42)
    print(f"Dataset sampled to {TARGET_SAMPLE_SIZE} rows.")
else:
    print(f"Using full dataset ({len(df)} rows).")

print(f"Final Dataset Shape: {df.shape}")
print(df.head())

In [2]:
def get_llm_reponse(prompt, model="gpt-3.5-turbo"):
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        # Mock response if no key is provided
        return "{\"predicted_stars\": 5, \"reasoning\": \"Positive keywords found.\", \"explanation\": \"Mock response (No API Key)\"}"
    
    try:
        from openai import OpenAI
        client = OpenAI(api_key=api_key)
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"API Error: {e}")
        return "{}"

In [3]:
# Strategy 1: Zero-shot Baseline
def prompt_zero_shot(review_text):
    return f"""Classify the sentiment of this review as a star rating from 1 to 5.
Review: {review_text}
Output: JSON with 'predicted_stars'."""

In [4]:
# Strategy 2: Few-shot
def prompt_few_shot(review_text):
    return f"""Examples:
Review: 'Loved it!' -> {{'predicted_stars': 5}}
Review: 'Terrible.' -> {{'predicted_stars': 1}}
Review: {review_text}
Output: JSON with 'predicted_stars'."""

In [5]:
# Strategy 3: Chain-of-Thought (CoT)
def prompt_cot(review_text):
    return f"""Analyze the review step-by-step.
1. Identify positive/negative keywords.
2. Determine the tone.
3. Assign a rating (1-5).
Review: {review_text}
Format: JSON with 'reasoning' and 'predicted_stars'."""

In [6]:
# Strategy 4: Strict JSON Enforcer
def prompt_strict_json(review_text):
    return f"""SYSTEM: You are a strict JSON data extractor.
USER: Extract the star rating (1-5) from this review.
Review: {review_text}
CRITICAL: Output ONLY valid JSON. No markdown, no preambles.
Schema: {{"predicted_stars": int, "explanation": str}}"""

In [7]:
# Strategy 5: Self-Correction / Retry Logic
def strategy_self_correction(review_text):
    # Step 1: Initial Prompt
    prompt = prompt_zero_shot(review_text)
    response = get_llm_reponse(prompt)
    try:
        data = json.loads(response)
        return data
    except json.JSONDecodeError:
        # Step 3: Retry
        retry_prompt = f"Previous response was invalid JSON. Fix this: {response}"
        return get_llm_reponse(retry_prompt)
    return json.loads(response)

In [8]:
# Strategy 6: Self-Consistency (Majority Vote)
def prompt_self_consistency(review_text):
    # In a real implementation, you set temperature > 0.7 to get diverse reasoning paths
    votes = []
    # print("  Running 3 consistency checks...")
    for _ in range(3):
        # response = get_llm_reponse(prompt_cot(review_text))
        # votes.append(extract_stars(response))
        votes.append(5) # Mock vote for structure
    
    # Logic to find majority
    if not votes: return 0
    final_vote = max(set(votes), key=votes.count)
    return {"predicted_stars": final_vote, "explanation": "Majority vote result"}

In [9]:
# --- EVALUATION LOOP ---

def extract_stars(response_str):
    # Helper to clean response and find JSON
    try:
        # If strictly JSON
        if isinstance(response_str, dict):
            return int(response_str.get('predicted_stars', 0))
        
        import json
        # Try to find JSON blob in string
        start = response_str.find('{')
        end = response_str.rfind('}') + 1
        if start != -1 and end != 0:
            json_str = response_str[start:end]
            data = json.loads(json_str)
            return int(data.get('predicted_stars', 0))
        return 0
    except:
        return 0

# Run evaluation on a small batch
print("Starting Evaluation Loop... (Running on first 5 rows for demo)")
results = []
test_df = df.head(5)

for i, row in test_df.iterrows():
    text = row['text']
    actual = row['stars']
    
    # -- USING STRATEGY 2: FEW-SHOT AS DEFAULT --
    prompt = prompt_few_shot(text)
    response = get_llm_reponse(prompt)
    predicted = extract_stars(response)
    
    is_correct = (predicted == actual)
    results.append(is_correct)
    
    print(f"Row {i}: Actual={actual} | Pred={predicted} | Correct={is_correct}")
    # print(f"   Raw Response: {response[:50]}...") # Debug

accuracy = sum(results) / len(results)
print(f"\nBatch Accuracy: {accuracy:.2%}")